# USDA Pipeline V3 — Reusable Functions

Sanitized export of the existing Databricks functions. Run definition cells only. Requires existing Delta tables in `workspace.usda_training`. Synthetic 2026 yields are not official USDA data.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

# Configuration only: does not create or overwrite Delta tables.
fingerprint_table = "workspace.usda_training.batch_source_fingerprints"
fingerprint_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("source_fingerprint", StringType(), False),
    StructField("registered_at", TimestampType(), False),
])


In [ ]:
import hashlib

import json

def calculate_batch_fingerprint(df):
    """
    Calculate a SHA-256 fingerprint for a training batch.

    Includes duplicate occurrences and ignores row order.
    Intended for small training batches.
    """

    required_columns = [
        "year",
        "state",
        "yield_bu_acre"
    ]

    rows = [
        row.asDict()
        for row in df.select(*required_columns).collect()
    ]

    # Canonical representation, including duplicates
    canonical_rows = sorted(
        json.dumps(
            row,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False
        )
        for row in rows
    )

    payload = json.dumps(
        canonical_rows,
        separators=(",", ":")
    )

    fingerprint = hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()

    return fingerprint


In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ==========================================
# 1. USDA BATCH VALIDATION
# ==========================================

def validate_usda_batch(df):

    invalid_condition = (
        F.col("year").isNull()
        | F.col("state").isNull()
        | (F.trim(F.col("state")) == "")
        | F.col("yield_bu_acre").isNull()
        | F.isnan("yield_bu_acre")
        | (F.col("yield_bu_acre") <= 0)
    )

    df_invalid = df.filter(invalid_condition)
    df_valid = df.filter(~invalid_condition)

    conflicting_keys = (
        df_valid
        .groupBy("year", "state")
        .agg(
            F.countDistinct("yield_bu_acre")
            .alias("distinct_yields")
        )
        .filter(F.col("distinct_yields") > 1)
        .select("year", "state")
    )

    df_conflicts = df_valid.join(
        conflicting_keys,
        ["year", "state"],
        "inner"
    )

    df_accepted = (
        df_valid
        .join(
            conflicting_keys,
            ["year", "state"],
            "left_anti"
        )
        .dropDuplicates(["year", "state"])
    )

    df_invalid = df_invalid.withColumn(
        "rejection_reason",
        F.lit("Missing or invalid field")
    )

    df_conflicts = df_conflicts.withColumn(
        "rejection_reason",
        F.lit("Conflicting yields for state-year")
    )

    df_quarantine = df_invalid.unionByName(
        df_conflicts
    )

    return df_accepted, df_quarantine


# ==========================================
# 2. RETRY-SAFE QUARANTINE WRITER
# ==========================================

def save_quarantine_once(batch_id, quarantine_df):

    quarantine_table = (
        "workspace.usda_training.delta_merge_quarantine_history"
    )

    if quarantine_df.limit(1).count() == 0:
        print("No quarantine records to save.")
        return

    key_columns = [
        "year", "state",
        "yield_bu_acre", "rejection_reason"
    ]

    duplicate_count = (
        quarantine_df
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if duplicate_count > 0:
        raise ValueError(
            "Duplicate quarantine source observations detected"
        )

    df_to_save = (
        quarantine_df
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("quarantined_at", F.current_timestamp())
    )

    target = DeltaTable.forName(
        spark,
        quarantine_table
    )

    (
        target.alias("target")
        .merge(
            df_to_save.alias("source"),
            """
            target.batch_id = source.batch_id
            AND target.year <=> source.year
            AND target.state <=> source.state
            AND target.yield_bu_acre <=> source.yield_bu_acre
            AND target.rejection_reason <=> source.rejection_reason
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("RETRY-SAFE QUARANTINE WRITE COMPLETED")


# ==========================================
# 3. CONDITIONAL DELTA MERGE
# ==========================================

def merge_accepted_usda(accepted_df):

    merge_table = (
        "workspace.usda_training.delta_merge_lab"
    )

    duplicate_keys = (
        accepted_df
        .groupBy("year", "state")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicate_keys == 0, (
        "Duplicate accepted state-year keys"
    )

    target = DeltaTable.forName(
        spark,
        merge_table
    )

    (
        target.alias("target")
        .merge(
            accepted_df.alias("source"),
            """
            target.year = source.year
            AND target.state = source.state
            """
        )
        .whenMatchedUpdate(
            condition="""
                NOT (
                    target.yield_bu_acre
                    <=> source.yield_bu_acre
                )
            """,
            set={
                "yield_bu_acre": "source.yield_bu_acre"
            }
        )
        .whenNotMatchedInsert(
            values={
                "year": "source.year",
                "state": "source.state",
                "yield_bu_acre": "source.yield_bu_acre",
                "period": "'YEAR'",
                "geo_level": "'STATE'",
                "commodity": "'CORN'",
                "data_item": "'CORN, GRAIN - YIELD, MEASURED IN BU / ACRE'",
                "domain": "'TOTAL'"
            }
        )
        .execute()
    )

    print("CONDITIONAL USDA DELTA MERGE COMPLETED")


# ==========================================
# 4. RETRY-SAFE BATCH AUDIT
# ==========================================

def save_batch_audit_once(df_audit, audit_table):

    duplicate_ids = (
        df_audit
        .groupBy("batch_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicate_ids == 0, (
        "Duplicate source batch IDs"
    )

    target = DeltaTable.forName(
        spark,
        audit_table
    )

    (
        target.alias("target")
        .merge(
            df_audit.alias("source"),
            "target.batch_id = source.batch_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("RETRY-SAFE AUDIT MERGE COMPLETED")


In [ ]:
from datetime import datetime, timezone
from delta.tables import DeltaTable
from pyspark.sql import functions as F

def register_batch_fingerprint(batch_id, fingerprint):
    """
    Register a new batch fingerprint or verify
    that an existing batch ID has the same fingerprint.
    """

    if not isinstance(batch_id, str) or not batch_id.strip():
        raise ValueError("Batch ID is required")

    if (
        not isinstance(fingerprint, str)
        or len(fingerprint) != 64
        or any(c not in "0123456789abcdef" for c in fingerprint)
    ):
        raise ValueError("Invalid SHA-256 fingerprint")

    existing = (
        spark.table(fingerprint_table)
        .filter(F.col("batch_id") == batch_id)
        .select("source_fingerprint")
        .collect()
    )

    if len(existing) > 1:
        raise ValueError("Duplicate fingerprint registry entries")

    if existing:
        if existing[0]["source_fingerprint"] != fingerprint:
            raise ValueError(
                "BATCH SOURCE MISMATCH: "
                "This batch ID is registered to different data."
            )

        print("EXISTING BATCH FINGERPRINT VERIFIED")
        return "VERIFIED"

    record = spark.createDataFrame(
        [(
            batch_id,
            fingerprint,
            datetime.now(timezone.utc)
        )],
        schema=fingerprint_schema
    )

    target = DeltaTable.forName(spark, fingerprint_table)

    (
        target.alias("target")
        .merge(
            record.alias("source"),
            "target.batch_id = source.batch_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    # Verify the persisted value
    saved = (
        spark.table(fingerprint_table)
        .filter(F.col("batch_id") == batch_id)
        .select("source_fingerprint")
        .collect()
    )

    assert len(saved) == 1
    assert saved[0]["source_fingerprint"] == fingerprint

    print("NEW BATCH FINGERPRINT REGISTERED")
    return "REGISTERED"


In [ ]:
from pyspark.sql import functions as F

def verify_batch_source_identity(batch_id, incoming_df):
    """
    Verify source identity before processing.

    This function is read-only:
    it does not register new fingerprints
    or modify any Delta tables.
    """

    fingerprint = calculate_batch_fingerprint(
        incoming_df
    )

    registry = (
        "workspace.usda_training.batch_source_fingerprints"
    )

    records = (
        spark.table(registry)
        .filter(F.col("batch_id") == batch_id)
        .select("source_fingerprint")
        .collect()
    )

    if len(records) > 1:
        raise ValueError(
            "Duplicate fingerprint registry entries"
        )

    if records:
        saved_fingerprint = records[0]["source_fingerprint"]

        if saved_fingerprint != fingerprint:
            raise ValueError(
                "BATCH SOURCE MISMATCH: "
                "Existing batch ID has different source data."
            )

        print("REGISTERED SOURCE IDENTITY VERIFIED")

        return {
            "batch_id": batch_id,
            "fingerprint": fingerprint,
            "status": "VERIFIED"
        }

    # Do not automatically trust historical batches
    audit_count = (
        spark.table(
            "workspace.usda_training.delta_merge_batch_audit"
        )
        .filter(F.col("batch_id") == batch_id)
        .count()
    )

    if audit_count > 0:
        print("HISTORICAL BATCH WITHOUT FINGERPRINT")
        print("Manual source verification required.")

        return {
            "batch_id": batch_id,
            "fingerprint": fingerprint,
            "status": "UNVERIFIED_HISTORICAL"
        }

    print("NEW BATCH SOURCE READY FOR REGISTRATION")

    return {
        "batch_id": batch_id,
        "fingerprint": fingerprint,
        "status": "NEW_SOURCE"
    }


In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime, timezone

duplicate_audit_table = (
    "workspace.usda_training.batch_duplicate_accounting"
)

def save_duplicate_accounting_once(batch_id, validation):
    """
    Save duplicate-aware batch accounting.

    Existing records must match exactly.
    A retry must not create duplicates.
    """

    count_columns = [
        "incoming_rows",
        "accepted_rows",
        "quarantined_rows",
        "duplicate_rows"
    ]

    values = {
        "incoming_rows": validation["incoming"],
        "accepted_rows": validation["accepted"],
        "quarantined_rows": validation["quarantined"],
        "duplicate_rows": validation["duplicates"]
    }

    assert values["incoming_rows"] == (
        values["accepted_rows"] +
        values["quarantined_rows"] +
        values["duplicate_rows"]
    )

    # Check whether accounting already exists
    existing = (
        spark.table(duplicate_audit_table)
        .filter(F.col("batch_id") == batch_id)
        .collect()
    )

    if len(existing) > 1:
        raise ValueError(
            "Duplicate accounting records detected"
        )

    if existing:
        for column in count_columns:
            if existing[0][column] != values[column]:
                raise ValueError(
                    f"BATCH ACCOUNTING MISMATCH: {column}"
                )

        print("EXISTING DUPLICATE ACCOUNTING VERIFIED")
        return "VERIFIED"

    record = {
        "batch_id": batch_id,
        **values,
        "recorded_at": datetime.now(timezone.utc)
    }

    df_record = spark.createDataFrame(
        [record],
        schema=spark.table(duplicate_audit_table).schema
    )

    target = DeltaTable.forName(
        spark,
        duplicate_audit_table
    )

    (
        target.alias("target")
        .merge(
            df_record.alias("source"),
            "target.batch_id = source.batch_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    # Verify the stored record
    saved = (
        spark.table(duplicate_audit_table)
        .filter(F.col("batch_id") == batch_id)
        .collect()
    )

    assert len(saved) == 1

    for column in count_columns:
        assert saved[0][column] == values[column]

    print("NEW DUPLICATE ACCOUNTING RECORD SAVED")

    return "REGISTERED"


In [ ]:
from pyspark.sql import functions as F

def inspect_usda_batch(batch_id, accepted_df):
    """
    Inspect existing batch records before processing.

    This function performs no writes.
    """

    target_table = (
        "workspace.usda_training.delta_merge_lab"
    )

    quarantine_table = (
        "workspace.usda_training.delta_merge_quarantine_history"
    )

    audit_table = (
        "workspace.usda_training.delta_merge_batch_audit"
    )

    # Check for a completed audit record
    audit_records = (
        spark.table(audit_table)
        .filter(F.col("batch_id") == batch_id)
    )

    audit_count = audit_records.count()

    # Check existing quarantine records
    quarantine_count = (
        spark.table(quarantine_table)
        .filter(F.col("batch_id") == batch_id)
        .count()
    )

    # Check whether accepted observations currently match
    # the target table
    source = accepted_df.select(
        "year", "state", "yield_bu_acre"
    ).dropDuplicates()

    target = spark.table(target_table).select(
        "year", "state", "yield_bu_acre"
    )

    matching_count = (
        source.alias("source")
        .join(
            target.alias("target"),
            (F.col("source.year") == F.col("target.year")) &
            (F.col("source.state") == F.col("target.state")) &
            F.col("source.yield_bu_acre").eqNullSafe(
                F.col("target.yield_bu_acre")
            ),
            "left_semi"
        )
        .count()
    )

    accepted_count = source.count()

    print("Batch ID:", batch_id)
    print("Audit records:", audit_count)
    print("Quarantine records:", quarantine_count)
    print("Matching accepted records:", matching_count)
    print("Expected accepted records:", accepted_count)

    if audit_count > 1:
        raise ValueError("Duplicate batch audit records")

    return {
        "batch_id": batch_id,
        "audit_records": audit_count,
        "quarantine_records": quarantine_count,
        "matching_accepted": matching_count,
        "expected_accepted": accepted_count
    }


In [ ]:
def determine_batch_recovery_action(inspection):
    """
    Classify a batch based on its inspection results.

    This function performs no Delta writes.
    """

    audit_count = inspection["audit_records"]
    quarantine_count = inspection["quarantine_records"]
    matching_count = inspection["matching_accepted"]
    accepted_count = inspection["expected_accepted"]

    if audit_count == 1:
        if matching_count != accepted_count:
            raise ValueError(
                "Completed batch does not match target data. "
                "Investigate before retrying."
            )

        print("Recovery decision: VERIFY_COMPLETED")
        return "VERIFY_COMPLETED"

    if audit_count > 1:
        raise ValueError("Duplicate audit records detected")

    # Existing quarantine or matching target data
    # may indicate an earlier partial attempt.
    if quarantine_count > 0 or matching_count > 0:
        print("Recovery decision: INVESTIGATE_PARTIAL")
        return "INVESTIGATE_PARTIAL"

    print("Recovery decision: PROCESS_NEW")
    return "PROCESS_NEW"


In [ ]:
def validate_usda_batch_v2(df):
    """
    Validate USDA records and account for exact
    duplicate observations separately.

    Performs no Delta table writes.
    """

    required_columns = [
        "year",
        "state",
        "yield_bu_acre"
    ]

    # Preserve only the required fields for this lab
    source = df.select(*required_columns)

    incoming_count = source.count()

    # Remove identical observations before validation
    unique_df = source.dropDuplicates(required_columns)

    unique_count = unique_df.count()

    duplicate_count = incoming_count - unique_count

    # Reuse the existing USDA validation rules
    accepted, quarantine = validate_usda_batch(
        unique_df
    )

    accepted_count = accepted.count()
    quarantine_count = quarantine.count()

    # All incoming observations must be accounted for
    assert incoming_count == (
        accepted_count +
        quarantine_count +
        duplicate_count
    ), "Batch reconciliation failed"

    print("Incoming:", incoming_count)
    print("Accepted:", accepted_count)
    print("Quarantined:", quarantine_count)
    print("Duplicates:", duplicate_count)

    print("DUPLICATE-AWARE BATCH VALIDATION PASSED")

    return {
        "accepted_df": accepted,
        "quarantine_df": quarantine,
        "incoming": incoming_count,
        "accepted": accepted_count,
        "quarantined": quarantine_count,
        "duplicates": duplicate_count
    }


In [ ]:
from pyspark.sql import functions as F

def verify_completed_usda_batch_v2(batch_id, incoming_df):
    """
    Verify a completed batch before allowing a skip.

    Read-only: does not modify Delta tables.
    """

    # 1. Verify the original source fingerprint
    identity = verify_batch_source_identity(
        batch_id, incoming_df
    )

    assert identity["status"] == "VERIFIED", (
        "Batch source identity is not verified"
    )

    # 2. Recalculate accepted, quarantined and duplicate counts
    validation = validate_usda_batch_v2(incoming_df)

    # 3. Compare with persisted duplicate accounting
    accounting_table = (
        "workspace.usda_training.batch_duplicate_accounting"
    )

    accounting = (
        spark.table(accounting_table)
        .filter(F.col("batch_id") == batch_id)
        .collect()
    )

    assert len(accounting) == 1

    for column, key in {
        "incoming_rows": "incoming",
        "accepted_rows": "accepted",
        "quarantined_rows": "quarantined",
        "duplicate_rows": "duplicates"
    }.items():
        assert accounting[0][column] == validation[key], (
            f"Accounting mismatch: {column}"
        )

    # 4. Verify accepted observations in the target
    target = spark.table(
        "workspace.usda_training.delta_merge_lab"
    )

    accepted = validation["accepted_df"]

    missing = (
        accepted.alias("source")
        .join(
            target.alias("target"),
            (F.col("source.year") == F.col("target.year")) &
            (F.col("source.state") == F.col("target.state")) &
            F.col("source.yield_bu_acre").eqNullSafe(
                F.col("target.yield_bu_acre")
            ),
            "left_anti"
        )
        .count()
    )

    assert missing == 0, "Accepted observations do not match"

    # 5. Verify all quarantined observations
    columns = [
        "year", "state", "yield_bu_acre",
        "rejection_reason"
    ]

    expected = validation["quarantine_df"].select(*columns)

    actual = (
        spark.table(
            "workspace.usda_training.delta_merge_quarantine_history"
        )
        .filter(F.col("batch_id") == batch_id)
        .select(*columns)
    )

    assert expected.exceptAll(actual).count() == 0
    assert actual.exceptAll(expected).count() == 0

    # 6. Verify completion audit
    audit_records = (
        spark.table(
            "workspace.usda_training.delta_merge_batch_audit"
        )
        .filter(F.col("batch_id") == batch_id)
        .collect()
    )

    assert len(audit_records) == 1

    audit = audit_records[0]

    assert audit["status"] == (
        "SUCCESS_WITH_QUARANTINE"
        if validation["quarantined"] > 0
        else "SUCCESS"
    )

    assert audit["incoming_rows"] == validation["incoming"]
    assert audit["accepted_rows"] == validation["accepted"]
    assert audit["quarantined_rows"] == validation["quarantined"]

    print("COMPLETED BATCH FULL RECONCILIATION PASSED")

    return True


In [ ]:
from pyspark.sql import functions as F
from datetime import datetime, timezone

def execute_usda_pipeline_v2(batch_id, incoming_df):
    """
    Duplicate-aware, fingerprint-verified
    USDA incremental ingestion workflow.
    """

    print("=" * 45)
    print("USDA PIPELINE V2")
    print("Batch ID:", batch_id)
    print("=" * 45)

    # 1. Verify the source fingerprint
    identity = verify_batch_source_identity(
        batch_id, incoming_df
    )

    if identity["status"] != "VERIFIED":
        raise ValueError(
            "Source fingerprint must be registered and verified"
        )

    # 2. Validate with duplicate-aware accounting
    validation = validate_usda_batch_v2(incoming_df)

    # 3. Verify persisted duplicate accounting
    accounting_table = (
        "workspace.usda_training.batch_duplicate_accounting"
    )

    records = (
        spark.table(accounting_table)
        .filter(F.col("batch_id") == batch_id)
        .collect()
    )

    if len(records) != 1:
        raise ValueError(
            "Exactly one duplicate accounting record is required"
        )

    record = records[0]

    count_mapping = {
        "incoming_rows": "incoming",
        "accepted_rows": "accepted",
        "quarantined_rows": "quarantined",
        "duplicate_rows": "duplicates"
    }

    for stored_column, result_key in count_mapping.items():
        if record[stored_column] != validation[result_key]:
            raise ValueError(
                f"Duplicate accounting mismatch: {stored_column}"
            )

    print("SOURCE AND ACCOUNTING VERIFIED")

    # 4. Inspect existing processing state
    inspection = inspect_usda_batch(
        batch_id,
        validation["accepted_df"]
    )

    action = determine_batch_recovery_action(inspection)

    if action == "VERIFY_COMPLETED":
        raise ValueError(
            "Batch already completed. "
            "Use a separate completed-batch reconciliation before skipping."
        )

    if action != "PROCESS_NEW":
        raise ValueError(
            "Potentially interrupted batch. Investigation required."
        )

    # 5. Save rejected observations
    save_quarantine_once(
        batch_id,
        validation["quarantine_df"]
    )

    # 6. MERGE accepted observations
    merge_accepted_usda(
        validation["accepted_df"]
    )

    # 7. Reconcile accepted observations
    target_table = (
        "workspace.usda_training.delta_merge_lab"
    )

    target = spark.table(target_table)

    accepted = validation["accepted_df"]

    missing_accepted = (
        accepted.alias("source")
        .join(
            target.alias("target"),
            (F.col("source.year") == F.col("target.year")) &
            (F.col("source.state") == F.col("target.state")) &
            F.col("source.yield_bu_acre").eqNullSafe(
                F.col("target.yield_bu_acre")
            ),
            "left_anti"
        )
        .count()
    )

    # 8. Reconcile quarantine records
    quarantine_table = (
        "workspace.usda_training.delta_merge_quarantine_history"
    )

    saved_quarantine = (
        spark.table(quarantine_table)
        .filter(F.col("batch_id") == batch_id)
    )

    columns = [
        "year", "state",
        "yield_bu_acre", "rejection_reason"
    ]

    expected = validation["quarantine_df"].select(*columns)
    actual = saved_quarantine.select(*columns)

    assert missing_accepted == 0
    assert expected.exceptAll(actual).count() == 0
    assert actual.exceptAll(expected).count() == 0

    assert (
        validation["incoming"]
        == validation["accepted"]
        + validation["quarantined"]
        + validation["duplicates"]
    )

    print("DUPLICATE-AWARE RECONCILIATION PASSED")

    # 9. Prepare and save the completion audit
    audit_table = (
        "workspace.usda_training.delta_merge_batch_audit"
    )

    audit_record = {
        "batch_id": batch_id,
        "processed_at": datetime.now(timezone.utc),
        "status": (
            "SUCCESS_WITH_QUARANTINE"
            if validation["quarantined"] > 0
            else "SUCCESS"
        ),
        "incoming_rows": validation["incoming"],
        "accepted_rows": validation["accepted"],
        "quarantined_rows": validation["quarantined"],
        "target_rows": target.count()
    }

    audit_df = spark.createDataFrame(
        [audit_record],
        schema=spark.table(audit_table).schema
    )

    save_batch_audit_once(audit_df, audit_table)

    print("USDA PIPELINE V2 COMPLETED")

    return {
        "batch_id": batch_id,
        "status": audit_record["status"],
        "incoming": validation["incoming"],
        "accepted": validation["accepted"],
        "quarantined": validation["quarantined"],
        "duplicates": validation["duplicates"]
    }


In [ ]:
from pyspark.sql import functions as F

def verify_quarantine_stage_recovery(batch_id, incoming_df):

    print("=" * 45)
    print("USDA CONTROLLED RECOVERY PREFLIGHT")
    print("Batch ID:", batch_id)
    print("=" * 45)

    # 1. Verify original registered source
    identity = verify_batch_source_identity(
        batch_id,
        incoming_df
    )

    if identity["status"] != "VERIFIED":
        raise ValueError(
            "Recovery blocked: source identity not verified."
        )

    # 2. Validate expected batch accounting
    validation = validate_usda_batch_v2(incoming_df)

    accounting_rows = (
        spark.table(
            "workspace.usda_training.batch_duplicate_accounting"
        )
        .filter(F.col("batch_id") == batch_id)
        .collect()
    )

    if len(accounting_rows) != 1:
        raise ValueError(
            "Recovery blocked: expected one accounting record."
        )

    accounting = accounting_rows[0]

    expected_counts = {
        "incoming_rows": validation["incoming"],
        "accepted_rows": validation["accepted"],
        "quarantined_rows": validation["quarantined"],
        "duplicate_rows": validation["duplicates"]
    }

    for column, expected in expected_counts.items():
        if accounting[column] != expected:
            raise ValueError(
                f"Recovery blocked: accounting mismatch for {column}."
            )

    # 3. Verify saved quarantine contents
    columns = [
        "year",
        "state",
        "yield_bu_acre",
        "rejection_reason"
    ]

    expected_quarantine = (
        validation["quarantine_df"]
        .select(*columns)
    )

    actual_quarantine = (
        spark.table(
            "workspace.usda_training.delta_merge_quarantine_history"
        )
        .filter(F.col("batch_id") == batch_id)
        .select(*columns)
    )

    missing = expected_quarantine.exceptAll(
        actual_quarantine
    ).count()

    unexpected = actual_quarantine.exceptAll(
        expected_quarantine
    ).count()

    if missing != 0 or unexpected != 0:
        raise ValueError(
            "Recovery blocked: quarantine contents mismatch."
        )

    # 4. Ensure none of the accepted keys are present
    target = spark.table(
        "workspace.usda_training.delta_merge_lab"
    )

    accepted_keys = (
        validation["accepted_df"]
        .select("year", "state")
        .distinct()
    )

    existing_accepted_keys = (
        target.join(
            accepted_keys,
            ["year", "state"],
            "inner"
        ).count()
    )

    if existing_accepted_keys != 0:
        raise ValueError(
            "Recovery blocked: accepted keys already exist in target."
        )

    # 5. Ensure no completion audit exists
    audit_count = (
        spark.table(
            "workspace.usda_training.delta_merge_batch_audit"
        )
        .filter(F.col("batch_id") == batch_id)
        .count()
    )

    if audit_count != 0:
        raise ValueError(
            "Recovery blocked: completion audit already exists."
        )

    # Require a nonempty quarantine stage for this recovery scenario
    if expected_quarantine.count() == 0:
        raise ValueError(
            "Recovery blocked: no quarantine stage to recover."
        )

    print("Source identity: VERIFIED")
    print("Accounting: VERIFIED")
    print("Quarantine contents: VERIFIED")
    print("Existing accepted keys:", existing_accepted_keys)
    print("Completion audit records:", audit_count)
    print("Recovery decision: SAFE_TO_RESUME_AFTER_QUARANTINE")

    return {
        "batch_id": batch_id,
        "status": "SAFE_TO_RESUME_AFTER_QUARANTINE",
        "validation": validation
    }


In [ ]:
def execute_usda_pipeline_v3(batch_id, incoming_df):
    """
    USDA Pipeline V3.

    Completed batches:
      Verify fingerprint, accounting, target,
      quarantine and audit, then skip.

    New batches:
      Delegate processing to Pipeline V2.

    Partially processed batches:
      Stop for investigation.
    """

    print("=" * 45)
    print("USDA PIPELINE V3")
    print("Batch ID:", batch_id)
    print("=" * 45)

    # Verify source identity first
    identity = verify_batch_source_identity(
        batch_id,
        incoming_df
    )

    if identity["status"] != "VERIFIED":
        raise ValueError(
            "A verified source fingerprint is required"
        )

    # Inspect the batch audit directly
    audit_table = (
        "workspace.usda_training.delta_merge_batch_audit"
    )

    audit_records = (
        spark.table(audit_table)
        .filter(F.col("batch_id") == batch_id)
        .select("status")
        .collect()
    )

    if len(audit_records) > 1:
        raise ValueError(
            "Duplicate completion audit records"
        )

    # Completed batch: reconcile before skipping
    if audit_records:
        status = audit_records[0]["status"]

        if status in ("SUCCESS", "SUCCESS_WITH_QUARANTINE"):

            verify_completed_usda_batch_v2(
                batch_id,
                incoming_df
            )

            print("BATCH VERIFIED AND SKIPPED")

            return {
                "batch_id": batch_id,
                "status": "SKIPPED_COMPLETED"
            }

        raise ValueError(
            f"Unexpected batch status: {status}"
        )

    # New or potentially interrupted batch:
    # V2 performs accounting and recovery checks
    print("NO COMPLETION AUDIT FOUND")
    print("Delegating to Pipeline V2")

    return execute_usda_pipeline_v2(
        batch_id,
        incoming_df
    )
